In [ ]:
# correlation
# price prediction 

# data combine 
# show correlation
# train linear regression on one year worth of data
# store linear regression model in flask
# predict estimated price based on that day's sentiment 
# display result 



In [99]:
#  data combine 

# get panel data first and add a date column for processing
import pandas as pd
from pandas_datareader import data
tickers = ['AAPL']

start_date = '2020-04-29'
end_date = '2021-10-10'

# User pandas_reader.data.DataReader to load data
panel_data = data.DataReader(tickers,'yahoo', start_date, end_date)
panel_data["date"] = panel_data.index
panel_data["date"] = pd.to_datetime(panel_data['date'])

# get reddit data or whatever for analysis and convert date type
reddit_data = pd.read_csv("./Reddit/cleaned_AAPL_sentiment.csv")
reddit_data["date"] = pd.to_datetime(reddit_data['date'])

# add day names for each of the two df 
from datetime import date
import calendar
for index, row in reddit_data.iterrows():
    reddit_data.at[index,"day_name"] = calendar.day_name[row["date"].weekday()]
    
for index, row in panel_data.iterrows():
    date = pd.to_datetime(row["date"].values[0])
    panel_data.at[index,"day_name"]  = calendar.day_name[date.weekday()]


In [100]:
reddit_data

for index, row in reddit_data.iterrows():
    if row['flair_sentiment'] == "NEGATIVE":
        reddit_data.at[index,"flair_sentiment_score"] = 0 - row['flair_sentiment_score']
    if row['finbert_sentiment'] == "negative":
        reddit_data.at[index,"finbert_score"] = 0 - row['flair_sentiment_score']
    if row['finbert_sentiment'] == "neutral":
        reddit_data.at[index,"finbert_score"] = 0

In [102]:
# groupby dates for panel data and sentiment data  
# note that weekends are missing for panel data
panel_groupby = panel_data.groupby(["date"]).mean()

In [103]:
reddit_groupby = reddit_data.groupby(["date"]).mean()

,likes,flair_sentiment_score,vader_score,finbert_score
date,,,,
2020-01-05,1.000000,-0.222682,0.041850,-0.202476
2020-01-06,1.000000,-0.263133,-0.000011,-0.287983
2020-01-07,1.000000,0.113954,0.060814,-0.136148
2020-01-08,1.000000,0.078797,0.053881,-0.097036
2020-01-09,1.290000,-0.332066,0.184554,-0.009753
...,...,...,...,...
2021-12-05,3.457364,-0.560480,0.281212,-0.006829
2021-12-06,2.973684,-0.515939,0.579839,0.155324
2021-12-07,7.166667,-0.302233,0.415522,0.033591


In [104]:
merge=pd.merge(reddit_groupby,panel_groupby, how='inner', left_index=True, right_index=True)

C:\Users\Shawn\anaconda3\lib\site-packages\pandas\core\reshape\merge.py:648: UserWarning: merging between different levels can give an unintended result (1 levels on the left,2 on the right)
  warnings.warn(msg, UserWarning)


In [105]:
merge.corr()


,likes,flair_sentiment_score,vader_score,finbert_score,"(Adj Close, AAPL)","(Close, AAPL)","(High, AAPL)","(Low, AAPL)","(Open, AAPL)","(Volume, AAPL)"
likes,1.000000,-0.181119,0.256752,0.151799,0.334455,0.332319,0.329207,0.333587,0.327397,-0.246790
flair_sentiment_score,-0.181119,1.000000,-0.391032,-0.162110,-0.468636,-0.468289,-0.476838,-0.472059,-0.479996,0.237497
vader_score,0.256752,-0.391032,1.000000,0.530927,0.526453,0.526025,0.526877,0.523649,0.524990,-0.335684
finbert_score,0.151799,-0.162110,0.530927,1.000000,0.253950,0.252908,0.252879,0.255387,0.257269,-0.283536
"(Adj Close, AAPL)",0.334455,-0.468636,0.526453,0.253950,1.000000,0.999959,0.997799,0.998205,0.995619,-0.458546
"(Close, AAPL)",0.332319,-0.468289,0.526025,0.252908,0.999959,1.000000,0.997916,0.998106,0.995634,-0.455175
"(High, AAPL)",0.329207,-0.476838,0.526877,0.252879,0.997799,0.997916,1.000000,0.997697,0.998385,-0.432386
"(Low, AAPL)",0.333587,-0.472059,0.523649,0.255387,0.998205,0.998106,0.997697,1.000000,0.997771,-0.473175
"(Open, AAPL)",0.327397,-0.479996,0.524990,0.257269,0.995619,0.995634,0.998385,0.997771,1.000000,-0.444980
"(Volume, AAPL)",-0.246790,0.237497,-0.335684,-0.283536,-0.458546,-0.455175,-0.432386,-0.473175,-0.444980,1.000000


In [78]:

# #  missing weekends - add weekend prices, leave it at friday 
# #  use the weighted sentiment 

# def weighted_weekends(data):
#     weighted_flair = weighted_vader = weighted_finbert = 0
#     new_df = []
#     for index, row in data.iterrows():
        
#         day = index.weekday()
#         if (day == 5): # saturday
#             weighted_flair += row['flair_sentiment_score'] * 0.2
#             weighted_vader += row['vader_score'] * 0.2
#             weighted_finbert += row['finbert_score'] * 0.2
#         elif (day == 6): # sunday
#             weighted_flair += row['flair_sentiment_score'] * 0.3
#             weighted_vader += row['vader_score'] * 0.3
#             weighted_finbert += row['finbert_score'] * 0.3
#         elif (day == 0): # monday
#             weighted_flair += row['flair_sentiment_score'] * 0.5
#             weighted_vader += row['vader_score'] * 0.5
#             weighted_finbert += row['finbert_score'] * 0.5
#             new_df.append([index, weighted_flair, weighted_vader, weighted_finbert])
#             weighted_flair = weighted_vader = weighted_finbert = 0
#         else:
#             new_df.append([index, row['flair_sentiment_score'], row['vader_score'], row['finbert_score']])
            
#     return new_df